In [ ]:
%pip cache purge

%pip install -r ../requirements.txt


In [ ]:
import os
import shutil

DIRECTORIES = [
    "../models", 
    "../data/raw/files"
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


In [ ]:
import mne
from mne.datasets import eegbci
from mne.io import read_raw_edf
from mne.io import concatenate_raws
from mne.time_frequency import EpochsTFR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import random
import logging
import json
from datetime import datetime
import pickle
from typing import List, Tuple, Dict, Optional

# Configure logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_random_data():
    """
    Load EEG data from a randomly selected subject and run.
    
    Returns:
        mne.io.Raw: MNE Raw object containing the selected EEG data
    """
    # Select a random subject (1-109)
    subject = random.randint(1, 109)
    
    # Define the runs for motor execution and motor imagery
    motor_execution_runs = [3, 5, 7, 9, 11, 13]  # Left/right hand or hands/feet
    motor_imagery_runs = [4, 6, 8, 10, 12, 14]   # Left/right hand or hands/feet
    
    # Choose randomly between motor execution and motor imagery
    if random.choice([True, False]):
        runs_list = motor_execution_runs
        task_type = "motor_execution"
    else:
        runs_list = motor_imagery_runs
        task_type = "motor_imagery"
    
    # Select a single random run from the selected list
    selected_run = random.choice(runs_list)
    
    # Check if it's left/right hand or hands/feet
    if selected_run in [3, 4, 7, 8, 11, 12]:
        paradigm = "left_right_hand"
    else:  # runs 5, 6, 9, 10, 13, 14
        paradigm = "hands_feet"
    
    logger.info(f"Selected subject: {subject}, run: {selected_run}")
    logger.info(f"Task type: {task_type}, paradigm: {paradigm}")
    
    # Load the data using MNE's built-in function
    raw_files = eegbci.load_data(subject, [selected_run])
    
    if not raw_files:
        raise ValueError(f"No data files found for subject {subject}, run {selected_run}")
    
    # Read and concatenate the raw EDF files
    raws = [read_raw_edf(f, preload=True) for f in raw_files]
    raw_data = concatenate_raws(raws)
    
    # Standardize channel names to international 10-20 system
    eegbci.standardize(raw_data)
    
    # Set EEG montage
    montage = mne.channels.make_standard_montage('standard_1005')
    raw_data.set_montage(montage)
    
    # Add metadata for reference
    # MNE's info doesn't allow custom keys, so we'll use info['subject_info'] and store our metadata
    # in a way that won't cause errors
    raw_data.info['subject_info'] = {'his_id': str(subject)}
    
    # Store additional metadata in a separate dictionary for later use
    metadata = {
        'task_type': task_type,
        'paradigm': paradigm,
        'run': selected_run
    }
    
    # We'll attach this to the raw object as an attribute
    raw_data.metadata = metadata
    
    return raw_data

def preprocess_data(raw_data, low_cutoff=8, high_cutoff=40, apply_notch=False):
    """
    Apply preprocessing steps to the raw EEG data.
    
    Args:
        raw_data (mne.io.Raw): Raw EEG data
        low_cutoff (float): Lower frequency cutoff for bandpass filter
        high_cutoff (float): Upper frequency cutoff for bandpass filter
        apply_notch (bool): Whether to apply notch filter for power line noise
        
    Returns:
        mne.io.Raw: Preprocessed EEG data
    """
    # Create a copy to avoid modifying the original data
    filter_data = raw_data.copy()
    
    # Apply bandpass filter
    logger.info(f"Applying bandpass filter ({low_cutoff}-{high_cutoff} Hz)...")
    filter_data.filter(low_cutoff, high_cutoff, fir_design='firwin')
    
    # Apply notch filter if requested (60Hz for US, 50Hz for Europe)
    if apply_notch:
        logger.info("Applying notch filter at 60Hz...")
        filter_data.notch_filter(freqs=[60], fir_design='firwin')
    
    return filter_data

def extract_epochs(filter_data, tmin=-1.0, tmax=4.0):
    """
    Extract epochs from the filtered data based on events.
    
    Args:
        filter_data (mne.io.Raw): Filtered EEG data
        tmin (float): Start time of the epoch relative to the event
        tmax (float): End time of the epoch relative to the event
        
    Returns:
        tuple: (mne.Epochs, dict) - Epochs object and event_id mapping
    """
    # Extract events from annotations
    events, event_id = mne.events_from_annotations(filter_data)
    
    # Map event IDs to more meaningful names
    # T0 -> rest, T1 -> class1 (left hand/both hands), T2 -> class2 (right hand/both feet)
    metadata = getattr(filter_data, 'metadata', {})
    paradigm = metadata.get('paradigm', '')
    
    if paradigm == 'left_right_hand':
        new_event_id = {
            'rest': event_id.get('T0', 0),
            'left_hand': event_id.get('T1', 0),
            'right_hand': event_id.get('T2', 0)
        }
    else:  # hands_feet
        new_event_id = {
            'rest': event_id.get('T0', 0),
            'both_hands': event_id.get('T1', 0),
            'both_feet': event_id.get('T2', 0)
        }
    
    # Remove any events with value 0 (not found)
    new_event_id = {k: v for k, v in new_event_id.items() if v != 0}
    
    logger.info(f"Event mapping: {new_event_id}")
    
    # Create epochs
    epochs = mne.Epochs(
        filter_data,
        events,
        event_id=new_event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=(None, 0),
        preload=True
    )
    
    logger.info(f"Created {len(epochs)} epochs with {len(epochs.ch_names)} channels")
    
    return epochs, new_event_id

def extract_features(epochs):
    """
    Extract features from epochs for machine learning.
    
    Args:
        epochs (mne.Epochs): Epochs object
        
    Returns:
        tuple: (X, y) - Features and labels
    """
    # Extract data and labels
    X = epochs.get_data()  # Shape: (n_epochs, n_channels, n_times)
    y = epochs.events[:, -1]  # Labels
    
    # Reshape for ML (flatten features)
    n_epochs, n_channels, n_times = X.shape
    X_flat = X.reshape(n_epochs, n_channels * n_times)
    
    # Normalization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat)
    
    logger.info(f"Extracted features: X shape {X.shape}, X_scaled shape {X_scaled.shape}")
    
    return X_scaled, y, scaler

def split_dataset(X, y, test_size=0.2, random_state=42):
    """
    Split the dataset into training and test sets.
    
    Args:
        X (np.ndarray): Features
        y (np.ndarray): Labels
        test_size (float): Proportion of the dataset to include in the test split
        random_state (int): Random seed for reproducibility
        
    Returns:
        tuple: (X_train, X_test, y_train, y_test) - Training and test sets
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    logger.info(f"Split dataset: X_train shape {X_train.shape}, X_test shape {X_test.shape}")
    logger.info(f"Class distribution in training set: {np.bincount(y_train)}")
    logger.info(f"Class distribution in test set: {np.bincount(y_test)}")
    
    return X_train, X_test, y_train, y_test

def save_preprocessed_data(X, y, scaler, raw_data, filter_data, epochs, save_dir='../models'):
    """
    Save the preprocessed data and metadata.
    
    Args:
        X (np.ndarray): Scaled features
        y (np.ndarray): Labels
        scaler (StandardScaler): Fitted scaler
        raw_data (mne.io.Raw): Raw EEG data
        filter_data (mne.io.Raw): Filtered EEG data
        epochs (mne.Epochs): Epochs object
        save_dir (str): Directory to save the data
    """
    # Create directory if it doesn't exist
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # Generate a timestamp for unique filenames
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save numpy arrays
    np.save(os.path.join(save_dir, f'X_preprocessed_{timestamp}.npy'), X)
    np.save(os.path.join(save_dir, f'y_labels_{timestamp}.npy'), y)
    
    # Create JSON-friendly preprocessing info
    # Get metadata from the raw_data.metadata attribute we created
    metadata = getattr(raw_data, 'metadata', {})
    
    preprocessing_info = {
        'subject': raw_data.info.get('subject_info', {}).get('his_id', 'unknown'),
        'run': metadata.get('run', 'unknown'),
        'task_type': metadata.get('task_type', 'unknown'),
        'paradigm': metadata.get('paradigm', 'unknown'),
        'channels': filter_data.info['ch_names'],
        'sampling_frequency': filter_data.info['sfreq'],
        'data_shape': {
            'epochs': X.shape[0],
            'features': X.shape[1]
        },
        'preprocessing_parameters': {
            'filter_low': filter_data.info.get('highpass', 0),
            'filter_high': filter_data.info.get('lowpass', 0),
            'scaling': 'StandardScaler'
        },
        'metadata': {
            'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
    }
    
    # Save JSON file
    json_path = os.path.join(save_dir, f'preprocessing_info_{timestamp}.json')
    with open(json_path, 'w') as f:
        json.dump(preprocessing_info, f, indent=4)
    
    # Save pickle file for objects that can't be JSON serialized
    pickle_info = {
        'scaling': scaler
    }
    with open(os.path.join(save_dir, f'preprocessing_objects_{timestamp}.pkl'), 'wb') as f:
        pickle.dump(pickle_info, f)
    
    logger.info(f"Saved preprocessed data with timestamp {timestamp}")
    
    return timestamp

def main():
    """Main function to run the full preprocessing pipeline."""
    try:
        # 1. Load random data
        raw_data = load_random_data()
        
        # 2. Apply preprocessing
        filter_data = preprocess_data(raw_data, low_cutoff=8, high_cutoff=40)
        
        # 3. Extract epochs
        epochs, event_id = extract_epochs(filter_data)
        
        # 4. Extract features
        X, y, scaler = extract_features(epochs)
        
        # 5. Split dataset
        X_train, X_test, y_train, y_test = split_dataset(X, y)
        
        # 6. Save preprocessed data
        timestamp = save_preprocessed_data(X, y, scaler, raw_data, filter_data, epochs)
        
        # 7. Print summary
        metadata = getattr(raw_data, 'metadata', {})
        print("\n===== Preprocessing Summary =====")
        print(f"Subject: {raw_data.info.get('subject_info', {}).get('his_id', 'unknown')}")
        print(f"Run: {metadata.get('run', 'unknown')}")
        print(f"Task: {metadata.get('task_type', 'unknown')}")
        print(f"Paradigm: {metadata.get('paradigm', 'unknown')}")
        print(f"Data shape: {X.shape}")
        print(f"Classes: {np.unique(y)}")
        print(f"Files saved with timestamp: {timestamp}")
        print("================================\n")
        
        return True
        
    except Exception as e:
        logger.error(f"Error in preprocessing pipeline: {str(e)}")
        raise e

if __name__ == "__main__":
    main()
